# AgriTech Precision Agriculture Platform
## Phase 1 (TODO-102): MobileNetV3-Small Transfer Learning on PlantVillage (12 Classes)

> **Target Hardware:** Google Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU)
> **Objective:** Fine-tune MobileNetV3-Small with outdoor-lighting augmentations, EarlyStopping (`patience=5`), evaluate on held-out test split (Acceptance Criteria: **Top-1 Macro F1 $\ge 0.94$**), and package model artifacts for download.

### Step 1: Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

!pip install -q scikit-learn matplotlib seaborn pillow

### Step 2: Download & Extract PlantVillage Dataset
Bu adımda PlantVillage veri kümesi doğrudan Colab sanal makinesine saniyeler içinde indirilir.

In [ ]:
import os
import shutil
from pathlib import Path

# Create data workspace in Colab /content
DATA_DIR = Path("/content/data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Option A: Clone/Download curated PlantVillage subset directly
!git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/pv_repo

# Locate raw directory
pv_color_dir = Path("/content/pv_repo/raw/color")
if not pv_color_dir.exists():
    print("Alternative path check...")
    # Fallback to direct download if needed
print(f"Raw dataset ready at: {pv_color_dir}")
print(f"Total raw folders: {len(list(pv_color_dir.iterdir()))}")

### Step 3: Run Deterministic 70/15/15 Stratified Split (TODO-101)
12 hedef sınıf filtrelenir, bozuk veya çift görseller elenir, ve deterministik `train/val/test` dizinleri oluşturulur.

In [ ]:
import hashlib
import random
from PIL import Image, UnidentifiedImageError

TARGET_12_CLASSES = [
    "Tomato___healthy",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Bacterial_spot",
    "Potato___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Corn_(maize)___healthy",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Apple___healthy",
    "Apple___Cedar_apple_rust",
]

def calculate_file_hash(file_path: Path) -> str:
    hasher = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()

def validate_image(image_path: Path, min_res=(224, 224)):
    try:
        with Image.open(image_path) as img:
            img.verify()
        with Image.open(image_path) as img:
            img.load()
            if img.mode != "RGB":
                return False, f"Non-RGB mode: {img.mode}"
            w, h = img.size
            if w < min_res[0] or h < min_res[1]:
                return False, f"Resolution {w}x{h} < {min_res}"
            return True, ""
    except Exception as e:
        return False, str(e)

def prepare_dataset_for_training(source_dir: Path, output_dir: Path, seed=42):
    splits = ["train", "val", "test"]
    for s in splits:
        for cls in TARGET_12_CLASSES:
            (output_dir / s / cls).mkdir(parents=True, exist_ok=True)
            
    seen_hashes = set()
    stats = {cls: {"train": 0, "val": 0, "test": 0, "rejected": 0} for cls in TARGET_12_CLASSES}
    
    for cls in TARGET_12_CLASSES:
        cls_dir = source_dir / cls
        if not cls_dir.exists():
            print(f"[!] Warning: Folder {cls} not found in {source_dir}")
            continue
            
        valid_files = []
        for f in sorted(list(cls_dir.iterdir())):
            if not f.is_file():
                continue
            is_valid, _ = validate_image(f)
            if not is_valid:
                stats[cls]["rejected"] += 1
                continue
            f_hash = calculate_file_hash(f)
            if f_hash in seen_hashes:
                stats[cls]["rejected"] += 1
                continue
            seen_hashes.add(f_hash)
            valid_files.append(f)
            
        # Deterministic Stratified 70/15/15 split
        rng = random.Random(seed)
        shuffled = valid_files.copy()
        rng.shuffle(shuffled)
        
        total = len(shuffled)
        train_end = int(round(total * 0.70))
        val_end = train_end + int(round(total * 0.15))
        
        train_files = shuffled[:train_end]
        val_files = shuffled[train_end:val_end]
        test_files = shuffled[val_end:]
        
        for split_name, files in [("train", train_files), ("val", val_files), ("test", test_files)]:
            for f in files:
                dst = output_dir / split_name / cls / f.name
                shutil.copy2(f, dst)
                stats[cls][split_name] += 1
                
    print("\n[+] Dataset Partitioning Completed!")
    train_total = sum(s["train"] for s in stats.values())
    val_total = sum(s["val"] for s in stats.values())
    test_total = sum(s["test"] for s in stats.values())
    print(f"    Train samples: {train_total}")
    print(f"    Val samples:   {val_total}")
    print(f"    Test samples:  {test_total}")
    print(f"    Total samples: {train_total + val_total + test_total}")
    return stats

stats = prepare_dataset_for_training(pv_color_dir, PROCESSED_DIR)

### Step 4: Model Architecture, Transforms & EarlyStopping
- **Backbone:** Pre-trained `MobileNetV3-Small` (ImageNet weights)
- **Augmentations:** `RandomHorizontalFlip`, `RandomRotation(degrees=30)`, `ColorJitter(brightness=0.2, contrast=0.2)`
- **Optimizer:** `AdamW(lr=1e-3, weight_decay=1e-4)` with `CosineAnnealingLR`
- **EarlyStopping:** `patience=5` monitoring validation loss

In [ ]:
import time
import json
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Outdoor lighting transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(str(PROCESSED_DIR / "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(str(PROCESSED_DIR / "val"), transform=eval_transform)
test_dataset = datasets.ImageFolder(str(PROCESSED_DIR / "test"), transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# Build Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features=in_features, out_features=12)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 18
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

class EarlyStopping:
    def __init__(self, patience=5, delta=0.0001, checkpoint_path="best_model.pth"):
        self.patience = patience
        self.delta = delta
        self.checkpoint_path = checkpoint_path
        self.counter = 0
        self.best_loss = float("inf")
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            print(f"[*] Validation loss improved ({val_loss:.4f}). Model saved!")
        else:
            self.counter += 1
            print(f"[!] Patience counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=5, checkpoint_path="/content/best_model.pth")
print(f"[*] Ready for training on: {device}")

### Step 5: Execute Training Loop (GPU Accelerated ~5-8 minutes)

In [ ]:
history = []
total_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    
    # --- Training Phase ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total
    scheduler.step()
    
    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            val_preds.extend(preds.cpu().tolist())
            val_targets.extend(labels.cpu().tolist())
            
    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    _, _, val_macro_f1, _ = precision_recall_fscore_support(val_targets, val_preds, average='macro', zero_division=0)
    epoch_time = time.time() - epoch_start
    
    record = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "train_acc": train_acc,
        "val_loss": avg_val_loss,
        "val_acc": val_acc,
        "val_macro_f1": float(val_macro_f1),
        "time_sec": epoch_time
    }
    history.append(record)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {avg_val_loss:.4f} | Val F1: {val_macro_f1:.4f} ({epoch_time:.1f}s)")
    
    early_stopping(avg_val_loss, model)
    if early_stopping.early_stop:
        print(f"[!] Early stopping triggered at epoch {epoch}.")
        break

print(f"\n[+] Training finished in {(time.time() - total_start)/60:.2f} minutes.")

### Step 6: Test Evaluation & Confusion Matrix
Held-out `test` kümesi üzerinde Top-1 Macro F1 skoru hesaplanır (Kabul kriteri: $\ge 0.94$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Load best model
best_model = models.mobilenet_v3_small(weights=None)
best_model.classifier[-1] = nn.Linear(in_features=in_features, out_features=12)
best_model.load_state_dict(torch.load("/content/best_model.pth"))
best_model = best_model.to(device)
best_model.eval()

test_preds, test_targets = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = best_model(imgs)
        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().tolist())
        test_targets.extend(labels.cpu().tolist())

class_names = test_dataset.classes
report_str = classification_report(test_targets, test_preds, target_names=class_names, digits=4)
report_dict = classification_report(test_targets, test_preds, target_names=class_names, output_dict=True)
print(report_str)

top1_acc = report_dict['accuracy']
macro_f1 = report_dict['macro avg']['f1-score']
print("=" * 60)
print(f"TEST TOP-1 ACCURACY: {top1_acc*100:.2f}%")
print(f"TEST MACRO F1-SCORE: {macro_f1:.4f}")
print(f"ACCEPTANCE CRITERIA (Macro F1 >= 0.94): {'PASS ✅' if macro_f1 >= 0.94 else 'FAIL ❌'}")
print("=" * 60)

# Confusion Matrix Plot
cm = confusion_matrix(test_targets, test_preds)
cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f"AgriTech MobileNetV3 Test Confusion Matrix (Macro F1: {macro_f1:.4f})", fontsize=13)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("/content/confusion_matrix.png", dpi=300)
plt.show()

# Save hyperparameters metadata
metadata = {
    "model_architecture": "MobileNetV3-Small",
    "num_classes": 12,
    "classes": class_names,
    "class_to_idx": test_dataset.class_to_idx,
    "top1_accuracy": float(top1_acc),
    "macro_f1": float(macro_f1),
    "acceptance_f1_met": bool(macro_f1 >= 0.94),
    "history": history,
    "per_class": report_dict
}
with open("/content/hyperparameters.json", "w") as f:
    json.dump(metadata, f, indent=2)

### Step 7: Zip Artifacts & Download to Your Computer
Bu hücre üretilen modeli ve raporları tek bir zip dosyası (`agritech_mobilenetv3_artifacts.zip`) haline getirip tarayıcınızdan bilgisayarınıza otomatik indirir.

In [ ]:
from google.colab import files

!zip -j /content/agritech_mobilenetv3_artifacts.zip /content/best_model.pth /content/hyperparameters.json /content/confusion_matrix.png

print("[+] Downloading artifacts archive to your computer...")
files.download("/content/agritech_mobilenetv3_artifacts.zip")